## Chat history with memory

### Load ENV file

In [1]:
from dotenv import load_dotenv
import os

load = load_dotenv('./../.env', override=True)

print(os.getenv('LANGSMITH_PROJECT'))

EALangchainTrainingsTest


### Create local/cloud LLM object

In [2]:
from langchain_ollama import ChatOllama
import os

ollama_cloud_llm = ChatOllama(
    base_url="http://localhost:11434/",  # Ollama cloud endpoint
    model="devstral-small-2:24b-cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
    temperature=0.5,
    max_tokens=350,
    headers={
        "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
    }
)

ollama_local_llm_1 = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest ",
    temperature=0.5,
    max_tokens=350
)

ollama_local_llm_2 = ChatOllama(
    base_url="http://localhost:11434/",
    model="qwen3.5:latest-256K",
    temperature=0.5,
    max_tokens=350
)

### Message history with ChatMessageHistory

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [4]:
template = ChatPromptTemplate.from_messages([
    ("placeholder", "{history}"),
    ("human", "{prompt}")
])

chain = template | ollama_cloud_llm | StrOutputParser()

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="prompt",
    history_messages_key="history")

session_id = "Shashank"

get_session_history(session_id).clear()

response1 = history.invoke(
    {"prompt": "What is the advantage of running the LLM in local machine? Just get me answer in bullet points and sub-bullet points. Keep the answer short."},
    config={"configurable": {"session_id": session_id}})

response2 = history.invoke(
    {"prompt": "How about for cloud?"},
    config={"configurable": {"session_id": session_id}})

print(response1)
print("\n--------------------------------------------------------------------\n")
print(response2)

- **Privacy & Security**
  - No data leaves your machine
  - Sensitive information stays local

- **Cost Efficiency**
  - No API call fees
  - One-time hardware investment

- **Customization**
  - Fine-tune models for specific tasks
  - Modify architecture as needed

- **Offline Access**
  - Works without internet
  - No dependency on external services

- **Performance Control**
  - Optimize for your hardware
  - Adjust inference speed/quality

- **No Rate Limits**
  - Unrestricted usage
  - No throttling by providers

- **Data Ownership**
  - Full control over inputs/outputs
  - No third-party logging

--------------------------------------------------------------------

- **Scalability**
  - Handle large workloads effortlessly
  - Auto-scaling based on demand

- **Cost-Effective for Sporadic Use**
  - Pay-as-you-go pricing
  - No upfront hardware costs

- **High Availability & Reliability**
  - Redundant systems minimize downtime
  - Professional maintenance & updates

- **Access to 

### Message history with SqlChatMessageHistory

In [5]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

template = ChatPromptTemplate.from_messages([
    ("placeholder", "{history}"),
    ("human", "{prompt}")
])

chain = template | ollama_cloud_llm | StrOutputParser()

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    return SQLChatMessageHistory(session_id=session_id, connection_string="sqlite:///./chat_history.db")

history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="prompt",
    history_messages_key="history")

session_id = "Shashank"

get_session_history(session_id).clear()

response1 = history.invoke(
    {"prompt": "What is the advantage of running the LLM in local machine? Just get me answer in bullet points and sub-bullet points. Keep the answer short."},
    config={"configurable": {"session_id": session_id}})

response2 = history.invoke(
    {"prompt": "How about for cloud?"},
    config={"configurable": {"session_id": session_id}})

print(response1)
print("\n--------------------------------------------------------------------\n")
print(response2)

/var/folders/5h/8jj3ndtd7xj7l2klkyy67wc00000gn/T/ipykernel_3710/775885844.py:21: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  get_session_history(session_id).clear()


- **Privacy and Security**
  - No data sent to third-party servers
  - Full control over sensitive information

- **Cost-Effective**
  - No API costs or subscription fees
  - One-time hardware investment

- **Offline Access**
  - Works without internet connection
  - Uninterrupted usage in low-connectivity areas

- **Customization & Control**
  - Modify models, fine-tune for specific tasks
  - Choose preferred hardware (GPU/CPU)

- **Performance & Latency**
  - Faster responses (no network delay)
  - Optimized for local hardware

- **No Rate Limits**
  - Unrestricted usage (no API call limits)
  - Run multiple queries simultaneously

- **Data Sovereignty**
  - Compliance with local regulations
  - No dependency on external providers

- **Long-Term Reliability**
  - Not affected by cloud service outages
  - Independent of vendor changes/policies

--------------------------------------------------------------------

- **Scalability**
  - Handle large workloads with ease
  - Dynamically a